In [78]:
from fastapi import FastAPI
from pydantic import BaseModel
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from dotenv import load_dotenv
import os

from python_backend import chunks_list

Loading Ollama Embeddings.It may take a few seconds or minutes ...
Split document TEST_001 into 1 chunks.
Total number of chunks: 28


In [79]:
load_dotenv()

False

In [80]:
# We will store our FAISS database in this variable
vector_store = None
FAISS_INDEX_PATH = "faiss_index"

In [92]:
def process_and_embed(doc_id: int, provided_id: str, code: str, additional_info: str, text: str):
    # 1. Create Document (This creates 'doc' locally inside the function)
    doc = Document(
        page_content=text,
        metadata={
            "original_id": doc_id,
            "provided_id": provided_id,
            "raw_code": code,
            "additional_info": additional_info
        }
    )

    # 2. Split Document into chunks (Now it can read 'doc' because they are in the same scope!)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100
    )



   # FIX: Use split_documents instead of create_documents
    chunks = splitter.split_documents([doc])
    print(f"[LangChain] Split document {provided_id} into {len(chunks)} chunks.")

    # 3. Store chunks in FAISS database safely
    if os.path.exists(FAISS_INDEX_PATH):
        print(f"[FAISS] Index found. Appending chunks to {FAISS_INDEX_PATH}...")
        vector_store = FAISS.load_local(
            FAISS_INDEX_PATH,
            embeddings,
            allow_dangerous_deserialization=True
        )
        vector_store.add_documents(chunks)
    else:
        print("[FAISS] No index found. Creating a fresh vector store...")
        vector_store = FAISS.from_documents(chunks, embeddings)

    # Save the progress back to disk
    vector_store.save_local(FAISS_INDEX_PATH)
    print("[FAISS] Index saved successfully.")

    return vector_store

In [96]:
print(chunks_list)
vector_store = process_and_embed(
    doc_id=1,
    provided_id="TEST_001",
    code="...",
    additional_info="...",
    text="..."
)

1
[LangChain] Split document TEST_001 into 1 chunks.
[FAISS] Index found. Appending chunks to faiss_index...
[FAISS] Index saved successfully.


In [97]:
print("Loading Ollama Embeddings.It may take a few seconds or minutes ...")
# Initialize the embedding model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    dimensions=1024,
)

Loading Ollama Embeddings.It may take a few seconds or minutes ...


In [99]:
print(vector_store.index_to_docstore_id)

{0: '375378ce-1b6c-4b73-963e-735818e158f9', 1: '02e79aa5-c99d-4138-b5cb-a333f05e3198', 2: 'cf046e13-4d9d-4f01-af6f-114867ff2e50', 3: '0f0d4d16-170c-4114-8dcc-a9bf735d8fd2', 4: '2bb2f22d-1535-498b-ab27-a82a3efe9f5c', 5: '09582a79-393a-4e1f-b08c-78d00d804bae', 6: '973296af-68c2-4d08-80cc-27a2807c6d5e', 7: '223161ca-643e-456b-a349-bd1339cb89b0', 8: '798e5cf8-5194-4633-a1e4-f95da0ca73f7', 9: 'f60f1f10-7a92-4a59-a697-fe88cf28519d', 10: '7888d943-9a6b-4e0a-ba0d-d01bc418dfbf', 11: '330f61cb-bde0-4b2d-96ba-bd451f9026a5'}


In [100]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # 'k' is the number of chunks to fetch
)

In [101]:
retriever.invoke("tell me about test")

[Document(id='330f61cb-bde0-4b2d-96ba-bd451f9026a5', metadata={'original_id': 1, 'provided_id': 'TEST_001', 'raw_code': '...', 'additional_info': '...'}, page_content='...'),
 Document(id='798e5cf8-5194-4633-a1e4-f95da0ca73f7', metadata={'original_id': 1, 'provided_id': 'DEFAULT_ID', 'raw_code': 'Print(hello wold in the era of agentic ai so that every body cn )'}, page_content='Get the code dont'),
 Document(id='f60f1f10-7a92-4a59-a697-fe88cf28519d', metadata={'original_id': 6, 'provided_id': None, 'raw_code': 'System.out.println("Hello from End-to-End!");'}, page_content='[Code Review Document]\nID: null\nCode Snippet: System.out.println("Hello from End-to-End!");\nContext: null')]